In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import os
import re
from shapely import wkt
from scipy.spatial.distance import cdist
from sklearn.metrics import r2_score
from scipy.stats import pearsonr

from datetime import datetime
from dateutil import parser
import pyproj
import geopandas as gpd
from shapely.geometry import box
from pyproj import CRS, Proj, Transformer

import warnings 
warnings.filterwarnings('ignore')

from tqdm import tqdm

import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F

from skimage.metrics import structural_similarity as compare_ssim
import matplotlib.gridspec as gridspec

import os, re, math, csv, json, random, numpy as np, xarray as xr
from collections import defaultdict
from datetime import datetime

from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import matplotlib.pyplot as plt

from pathlib import Path

In [ ]:
# Change from 5km to 1km to get 1km predictions
trop_dir = Path('/media/mqy5198/easystore/TROPOMI_vs_NO2/5km_grid_features')
omi_dir  = Path('/media/mqy5198/easystore/OMI_vs_NO2/5km_grid_features_v1')

def parse_date_key(p: Path):
    """Return a date key (YYYY-MM-DD) from filename stem supporting 20181001 and 2018-10-01."""
    s = p.stem
    for fmt in ("%Y%m%d", "%Y-%m-%d"):
        try:
            return datetime.strptime(s, fmt).date().isoformat()  
        except ValueError:
            pass
    raise ValueError(f"Unrecognized date format in filename: {p.name}")

trop_files = [p for p in trop_dir.iterdir() if p.suffix == ".nc"]
omi_files  = [p for p in omi_dir.iterdir()  if p.suffix == ".nc"]

trop_by_date = {parse_date_key(p): p for p in trop_files}
omi_by_date  = {parse_date_key(p): p for p in omi_files}

all_dates = sorted(set(trop_by_date) | set(omi_by_date))

all_files = []

for d in all_dates:
    if d in trop_by_date and d in omi_by_date:
        all_files.append(str(trop_by_date[d]))
    elif d in trop_by_date:
        all_files.append(str(trop_by_date[d]))
    else:
        all_files.append(str(omi_by_date[d]))

In [ ]:
sample_size = min(1000, len(all_files))
selected_files = random.sample(all_files, sample_size)

all_files = selected_files

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

date_pat = re.compile(r'(20\d{2}-\d{2}-\d{2})')
def extract_year(fname):
    m = date_pat.search(fname)
    if m: return int(m.group(1)[:4])
    b = os.path.basename(fname)
    return int(b[:4]) if b[:4].isdigit() else None

In [ ]:
class BasicBlockGN(nn.Module):
    def __init__(self, ch, groups=8):
        super().__init__()
        g = min(groups, ch)
        self.conv1 = nn.Conv2d(ch, ch, 3, padding=1, bias=False)
        self.gn1   = nn.GroupNorm(g, ch)
        self.relu  = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(ch, ch, 3, padding=1, bias=False)
        self.gn2   = nn.GroupNorm(g, ch)
    def forward(self, x):
        idt = x
        x = self.relu(self.gn1(self.conv1(x)))
        x = self.gn2(self.conv2(x))
        return self.relu(x + idt)

class ResNetGN(nn.Module):
    def __init__(self, in_channels, base=64, blocks=10, groups=8):
        super().__init__()
        g = min(groups, base)
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, base, 3, padding=1, bias=False),
            nn.GroupNorm(g, base),
            nn.ReLU(inplace=True),
        )
        self.blocks = nn.Sequential(*[BasicBlockGN(base, groups=groups) for _ in range(blocks)])
        self.head   = nn.Conv2d(base, 1, 1)
    def forward(self, x):
        return self.head(self.blocks(self.stem(x)))

def _gaussian_window_2d(window_size: int, sigma: float, device):
    ax = torch.arange(window_size, device=device) - window_size // 2
    g1d = torch.exp(-(ax**2) / (2*sigma*sigma))
    g1d = g1d / g1d.sum()
    g2d = torch.outer(g1d, g1d)
    return g2d

class SSIMLoss(nn.Module):
    def __init__(self, window_size=11, sigma=1.5, C1=(0.01**2), C2=(0.03**2)):
        super().__init__()
        self.window_size = window_size
        self.sigma = sigma
        self.C1 = C1
        self.C2 = C2
        self.register_buffer('window', None, persistent=False)

    def _prep_window(self, device):
        if self.window is None or self.window.device != device:
            w = _gaussian_window_2d(self.window_size, self.sigma, device)
            self.window = w.unsqueeze(0).unsqueeze(0)  

    def forward(self, x, y):
        self._prep_window(x.device)
        padding = self.window_size // 2
        conv = lambda z: F.conv2d(z, self.window, padding=padding, groups=1)

        mu_x = conv(x); mu_y = conv(y)
        mu_x2 = mu_x * mu_x; mu_y2 = mu_y * mu_y; mu_xy = mu_x * mu_y
        sigma_x2 = conv(x*x) - mu_x2
        sigma_y2 = conv(y*y) - mu_y2
        sigma_xy = conv(x*y) - mu_xy

        ssim_map = ((2*mu_xy + self.C1) * (2*sigma_xy + self.C2)) / ((mu_x2 + mu_y2 + self.C1) * (sigma_x2 + sigma_y2 + self.C2) + 1e-12)
        return (1.0 - ssim_map.clamp(0,1)).mean()

def normalize_channels(arrC, meanC, stdC):
    arr = arrC.astype(np.float32, copy=False)
    for c in range(arr.shape[0]):
        s = stdC[c] if np.isfinite(stdC[c]) and stdC[c] > 1e-8 else 1.0
        m = meanC[c] if np.isfinite(meanC[c]) else 0.0
        arr[c] = (arr[c] - m) / s
    return arr

def per_fold_global_norm(files: List[str], covariate_names: List[str],
                         sample_cap=2000, pixel_cap=400_000, clip=5e3, seed=42):
    rng = np.random.default_rng(seed)
    picks = list(files)
    if len(picks) > sample_cap:
        picks = [picks[i] for i in rng.choice(len(picks), size=sample_cap, replace=False)]

    with xr.open_dataset(os.path.join(data_dir, picks[0])) as ds0:
        pass
    C = 1 + len(covariate_names)  
    sumC   = np.zeros(C, np.float64)
    sumC2  = np.zeros(C, np.float64)
    countC = np.zeros(C, np.float64)

    for f in picks:
        with xr.open_dataset(os.path.join(data_dir, f)) as ds:
            no2 = np.nan_to_num(ds['no2'].values, nan=0.0).astype(np.float32)
            cov = [np.nan_to_num(ds[v].values, nan=0.0).astype(np.float32) for v in covariate_names]
            arr = np.stack([no2] + cov, axis=0)
            vec = arr.reshape(C, -1)
            if pixel_cap and vec.shape[1] > pixel_cap:
                idx = rng.choice(vec.shape[1], size=pixel_cap, replace=False)
                vec = vec[:, idx]
            if clip is not None:
                np.clip(vec, -clip, clip, out=vec)
            sumC  += vec.sum(axis=1, dtype=np.float64)
            sumC2 += np.square(vec, dtype=np.float64).sum(axis=1, dtype=np.float64)
            countC += vec.shape[1]
    if countC.sum() == 0 or np.any(countC == 0):
        return np.zeros((C,), np.float32), np.ones((C,), np.float32)
    meanC = (sumC / countC).astype(np.float32)
    varC  = (sumC2 / countC) - np.square(meanC.astype(np.float64))
    varC  = np.maximum(varC, 1e-8).astype(np.float32)
    stdC  = np.sqrt(varC).astype(np.float32)
    return meanC, stdC

def blockwise_calibrate_simple(pred, obs, min_pts=8):
    m = np.isfinite(obs)
    if m.sum() < min_pts:
        return pred
    x = pred[m].astype(np.float64)
    y = obs[m].astype(np.float64)
    Sx, Sy = x.sum(), y.sum()
    Sxx, Sxy = (x*x).sum(), (x*y).sum()
    N = float(m.sum())
    lam = 1e-2
    XtX = np.array([[Sxx + lam, Sx],
                    [Sx,  N   + lam]], dtype=np.float64)
    Xty = np.array([Sxy, Sy], dtype=np.float64)
    try:
        a, b0 = np.linalg.solve(XtX, Xty)
        out = (a * pred + b0).astype(np.float32)
        return out
    except Exception:
        med = float(np.median(y - x))
        return (pred + med).astype(np.float32)

def masked_metrics(obs2d, pred2d, mask2d, min_pts=3, min_val=0.0, max_val=300.0):
    m = np.isfinite(obs2d) & np.isfinite(pred2d) & mask2d
    if m.sum() < min_pts: 
        return np.nan, np.nan, np.nan, 0
    o = np.clip(obs2d[m], min_val, max_val).astype(np.float32)
    p = np.clip(pred2d[m], min_val, max_val).astype(np.float32)
    rmse = float(np.sqrt(np.mean((p - o)**2)))
    mae  = float(np.mean(np.abs(p - o)))
    ss_res = float(np.sum((o - p)**2))
    ss_tot = float(np.sum((o - o.mean())**2))
    r2 = 1.0 - ss_res/ss_tot if ss_tot > 0 else np.nan
    return rmse, r2, mae, int(m.sum())

In [ ]:
K_FOLDS = 10
EPOCHS = 1000
PATIENCE = 20
LR = 1e-4
WEIGHT_DECAY = 1e-4
USE_AMP = True
GRAD_CLIP = 1.0
EMA_DECAY = 0.995

ALPHA = 1 
GAMMA = 1  
DELTA = 1  

MIN_STATION_POINTS = 10
MAX_SAT_NAN_RATIO  = 0.50  

NORM_SAMPLE_CAP = 2000
NORM_PIXEL_CAP  = 400_000
NORM_CLIP       = 5e3

with xr.open_dataset(os.path.join(data_dir, all_files[0])) as ds0:
    all_vars = list(ds0.data_vars)

drop_names = {'no2', 'stations', 'stations_interpolated'}
covariate_names = [v for v in all_vars if v not in drop_names]
in_ch = 1 + len(covariate_names)

In [ ]:
rng = np.random.default_rng(123)
perm = list(all_files)
rng.shuffle(perm)
fold_sizes = [len(perm)//K_FOLDS + (1 if i < len(perm) % K_FOLDS else 0) for i in range(K_FOLDS)]
folds = []
start = 0
for fs in fold_sizes:
    folds.append(perm[start:start+fs])
    start += fs

def files_minus(all_files, subset):
    sset = set(subset)
    return [f for f in all_files if f not in sset]

ssim_loss_fn = SSIMLoss().to(device)
mse_loss = nn.MSELoss(reduction='none').to(device)
mse_full = nn.MSELoss(reduction='mean').to(device)

def make_model():
    return ResNetGN(in_channels=in_ch, base=64, blocks=10, groups=8).to(device)

def no2_channel_minmax(x):
    a = x[0:1]
    amin = float(np.nanmin(a))
    amax = float(np.nanmax(a))
    denom = (amax - amin) if (amax - amin) > 1e-6 else 1.0
    return (a - amin)/denom

def train_epoch(model, optimizer, meanC, stdC, train_files):
    model.train()
    scaler = GradScaler(enabled=USE_AMP)
    total_loss, steps = 0.0, 0

    for f in train_files:
        with xr.open_dataset(os.path.join(data_dir, f)) as ds:
            no2 = ds['no2'].values
            stn = ds['stations'].values
            stn_interp = ds['stations_interpolated'].values

            station_mask = np.isfinite(stn)
            valid_points = int(station_mask.sum())
            sat_nan_ratio = float(np.isnan(no2).sum()) / no2.size

            if (sat_nan_ratio > MAX_SAT_NAN_RATIO) or (valid_points < MIN_STATION_POINTS):
                continue

            no2_in = np.nan_to_num(no2, nan=0.0).astype(np.float32)
            no2_in = np.log1p(np.maximum(no2_in, 0.0))  

            cov = []
            for v in covariate_names:
                vv = np.nan_to_num(ds[v].values, nan=0.0).astype(np.float32)
                name = v.lower()
                if ('nei' in name) or ('emis' in name) or ('emission' in name):
                    vv = np.sqrt(np.maximum(vv, 0.0))
                cov.append(vv)

            arrC = np.stack([no2_in] + cov, axis=0)
            arrC = normalize_channels(arrC, meanC, stdC)

            gt_interp = np.nan_to_num(stn_interp, nan=0.0).astype(np.float32)
            msk = station_mask.astype(np.float32)

            x_in = torch.tensor(arrC).unsqueeze(0).to(device)               
            y_interp = torch.tensor(gt_interp).unsqueeze(0).unsqueeze(0).to(device)  
            mask_t = torch.tensor(msk).unsqueeze(0).unsqueeze(0).to(device)          

            optimizer.zero_grad(set_to_none=True)
            with autocast(enabled=USE_AMP):
                pred = model(x_in)

                mse_grid = mse_loss(pred, y_interp)                  
                masked_mse = (mse_grid * mask_t).sum() / (mask_t.sum() + 1e-8)

                spatial_consistency = mse_full(pred, y_interp)

                no2_01 = torch.tensor(no2_channel_minmax(arrC), dtype=torch.float32, device=device).unsqueeze(0)

                p = pred.clone()
                p_min = p.amin(dim=(-2,-1), keepdim=True)
                p_max = p.amax(dim=(-2,-1), keepdim=True)
                p01 = (p - p_min) / torch.clamp(p_max - p_min, min=1e-6)
                ssim_term = ssim_loss_fn(p01, no2_01)

                loss = ALPHA*masked_mse + GAMMA*spatial_consistency + DELTA*ssim_term

            if not torch.isfinite(loss):
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()

            total_loss += float(loss.detach().cpu().item()); steps += 1

    return (total_loss/max(1,steps)), steps

def evaluate_fold(ema_state, meanC, stdC, val_files):
    model = make_model()
    model.load_state_dict(ema_state, strict=True)
    model.eval()

    obs_list, pred_list = [], []
    n_points = 0

    with torch.no_grad():
        for f in val_files:
            with xr.open_dataset(os.path.join(data_dir, f)) as ds:
                no2 = ds['no2'].values
                stn = ds['stations'].values
                stn_interp = ds['stations_interpolated'].values

                station_mask = np.isfinite(stn)
                valid_points = int(station_mask.sum())
                sat_nan_ratio = float(np.isnan(no2).sum()) / no2.size

                if (sat_nan_ratio > MAX_SAT_NAN_RATIO) or (valid_points < MIN_STATION_POINTS):
                    continue

                no2_in = np.nan_to_num(no2, nan=0.0).astype(np.float32)
                no2_in = np.log1p(np.maximum(no2_in, 0.0))

                cov = []
                for v in covariate_names:
                    vv = np.nan_to_num(ds[v].values, nan=0.0).astype(np.float32)
                    name = v.lower()
                    if ('nei' in name) or ('emis' in name) or ('emission' in name):
                        vv = np.sqrt(np.maximum(vv, 0.0))
                    cov.append(vv)

                arrC = np.stack([no2_in] + cov, axis=0)
                arrC = normalize_channels(arrC, meanC, stdC)

                pred = model(torch.tensor(arrC).unsqueeze(0).to(device)).squeeze().cpu().numpy().astype(np.float32)
                pred[pred < 0] = 0.0

                pred_cal = blockwise_calibrate_simple(pred, stn, min_pts=8)

                rmse, r2, mae, n = masked_metrics(stn, pred_cal, np.isfinite(stn),
                                                  min_pts=MIN_STATION_POINTS)
                if n >= MIN_STATION_POINTS and np.isfinite(rmse):
                    obs_list.append(stn[np.isfinite(stn)])
                    pred_list.append(pred_cal[np.isfinite(stn)])
                    n_points += n

    if n_points < MIN_STATION_POINTS:
        return np.nan, np.nan, np.nan, 0

    obs_all  = np.concatenate([o.flatten() for o in obs_list]).astype(np.float32)
    pred_all = np.concatenate([p.flatten() for p in pred_list]).astype(np.float32)
    
    rmse = float(np.sqrt(np.mean((pred_all - obs_all) ** 2)))
    mae  = float(np.mean(np.abs(pred_all - obs_all)))
    ss_res = float(np.sum((obs_all - pred_all) ** 2))
    ss_tot = float(np.sum((obs_all - obs_all.mean()) ** 2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

    return rmse, mae, r2, n_points

In [ ]:
metrics_records = []

for k in range(K_FOLDS):
    val_files = folds[k]
    train_files = files_minus(all_files, val_files)
    print(f"\n===== Fold {k+1}/{K_FOLDS} | train={len(train_files)} | val={len(val_files)} =====")

    meanC, stdC = per_fold_global_norm(train_files, covariate_names,
                                       sample_cap=NORM_SAMPLE_CAP,
                                       pixel_cap=NORM_PIXEL_CAP,
                                       clip=NORM_CLIP, seed=42)

    model = make_model()
    ema   = make_model()
    ema.load_state_dict(model.state_dict())
    for p in ema.parameters(): p.requires_grad = False

    opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = GradScaler(enabled=USE_AMP)

    best_val = float('inf')
    no_imp   = 0

    for epoch in range(1, EPOCHS+1):
        train_loss, nsteps = train_epoch(model, opt, meanC, stdC, train_files)


        with torch.no_grad():
            msd, esd = model.state_dict(), ema.state_dict()
            for kk in esd.keys():
                esd[kk].copy_(EMA_DECAY * esd[kk] + (1.0-EMA_DECAY) * msd[kk])


        metric = train_loss
        if metric < best_val - 1e-6:
            best_val = metric
            no_imp = 0
        else:
            no_imp += 1

        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:4d} | train_loss={train_loss:.4f} | no_imp={no_imp}")

        if no_imp >= PATIENCE:
            break

    ema_state = {k:v.detach().clone() for k,v in ema.state_dict().items()}


    rmse, mae, r2, n_pts = evaluate_fold(ema_state, meanC, stdC, val_files)
    print(f"[Fold {k}] n={n_pts}  RMSE={rmse:.3f}  MAE={mae:.3f}  R2={r2:.3f}")
    metrics_records.append({"fold":int(k), "n_val_points":int(n_pts),
                            "RMSE":float(rmse), "MAE":float(mae), "R2":float(r2)})

In [14]:
metrics_df = pd.DataFrame(metrics_records)
print("\n=== 10-fold CV summary (mean ± sd) ===")
print(metrics_df[['RMSE','MAE','R2']].agg(['mean','std']).round(3))

Device: cuda
In channels: 242 (1 no2 + 241 covariates)

===== Fold 1/10 | train=900 | val=100 =====
  Epoch    1 | train_loss=0.0000 | no_imp=0
  Epoch    5 | train_loss=0.0000 | no_imp=4
  Epoch   10 | train_loss=0.0000 | no_imp=9
  Epoch   15 | train_loss=0.0000 | no_imp=14
  Epoch   20 | train_loss=0.0000 | no_imp=19
[Fold 0] n=5193  RMSE=2.668  MAE=2.280  R2=0.816

===== Fold 2/10 | train=900 | val=100 =====
  Epoch    1 | train_loss=0.0000 | no_imp=0
  Epoch    5 | train_loss=0.0000 | no_imp=4
  Epoch   10 | train_loss=0.0000 | no_imp=9
  Epoch   15 | train_loss=0.0000 | no_imp=14
  Epoch   20 | train_loss=0.0000 | no_imp=19
[Fold 1] n=5257  RMSE=2.717  MAE=2.314  R2=0.729

===== Fold 3/10 | train=900 | val=100 =====
  Epoch    1 | train_loss=0.0000 | no_imp=0
  Epoch    5 | train_loss=0.0000 | no_imp=4
  Epoch   10 | train_loss=0.0000 | no_imp=9
  Epoch   15 | train_loss=0.0000 | no_imp=14
  Epoch   20 | train_loss=0.0000 | no_imp=19
[Fold 2] n=5207  RMSE=2.684  MAE=2.276  R2=0.7